# Replicating Tessler et al. (2024) Fig. 4C — minority weight in Habermas Machine group statements

Pipeline: Sentence-T5 embeddings → per-question position axis (negating → affirming) → position component scores →
convex regression of group-statement scores on constituent opinion scores, per level of division → minority weight
(sum of minority coefficients), averaged over levels.

Method (SM 5.1, 5.4.1): Sentence-T5 embeddings; per-question position axis = unit vector from the embedding of
"No, I disagree. <negating statement>" to "Yes, I agree. <affirming statement>"; position score = projection onto that axis;
minority = the side of neutral with fewer pre-deliberation ratings, neutral opinions count as non-minority; convex regression
(weights >= 0, sum = 1) of statement scores on the group's opinion scores, one regression per (group size, minority size) level,
minority weight = sum of minority coefficients, averaged over levels weighted by number of rounds.

This notebook is run once per embedding model. `HM_EMB_DIR` picks the cache; the results go to
`results/<model>/{fig4c_primary.json, sensitivity.csv, summary.json}`, which are the only results
the report reads. `python reproduce.py results` runs all four models and saves the executed
notebook beside them.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hm_fig4c import pipeline as P
from hm_fig4c.analysis import assign_minority
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

EMB_DIR = os.environ.get("HM_EMB_DIR", "../embeddings/st5-base")   # embedding cache to use
AXIS_METHOD = os.environ.get("HM_AXIS_METHOD", "unit")               # 'unit' (SM eq. 5: projection on the unit axis) or 'affine'
N_BOOT = int(os.environ.get("HM_N_BOOT", "500"))
MODEL_TAG = os.path.basename(EMB_DIR.rstrip("/"))
OUT_DIR = f"../results/{MODEL_TAG}"; os.makedirs(OUT_DIR, exist_ok=True)
print(EMB_DIR, AXIS_METHOD, N_BOOT)

## 1. Score all texts on the position axis

In [ ]:
ENDPOINTS = os.environ.get("HM_ENDPOINTS", "prefixed")  # 'prefixed' (SM: generic + question-specific), 'plain', 'generic'
opinions, statements, questions, candidates = P.score_all("../prepared", EMB_DIR, method=AXIS_METHOD, endpoint_style=ENDPOINTS)
cov = pd.DataFrame({"opinions_scored": opinions.groupby("cohort")["score"].apply(lambda s: s.notna().mean()),
                    "initial_scored": statements.groupby("cohort")["initial_score"].apply(lambda s: s.notna().mean()),
                    "revised_scored": statements.groupby("cohort")["revised_score"].apply(lambda s: s.notna().mean()),
                    "candidates_scored": candidates.groupby("cohort")["score"].apply(lambda s: s.notna().mean()),
                    "n_rounds": statements.groupby("cohort").size(), "n_prereg_rounds": statements.groupby("cohort")["prereg"].sum()})
cov.round(3)

Each opinion now carries a `score`: the projection of its embedding on its question's position axis.

In [ ]:
opinions[["metadata.version", "launch_id", "round_id", "participant_id", "pre_rating", "score", "opinion_text"]].head()

The candidate statements are scored on the same axis, one row per candidate the mediator produced.

In [ ]:
candidates[["launch_id", "round_id", "phase", "is_winner", "score", "candidate_text"]].head()

## 2. Fig. 4A check — opinion position score vs pre-deliberation position rating (paper: r = 0.64)

In [ ]:
rows = {c: P.fig4a_correlation(P.select_cohort(opinions, c)) for c in ["cohort1", "cohort2", "cohort3", "cohorts_1_3", "cohort4", "training", "vca"]}
fig4a = pd.DataFrame(rows).T; fig4a

In [ ]:
d = P.select_cohort(opinions, "cohorts_1_3").dropna(subset=["score", "pre_rating"])
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(d["pre_rating"] + np.random.uniform(-.15, .15, len(d)), d["score"], s=4, alpha=.25)
means = d.groupby("pre_rating")["score"].mean()
ax.plot(means.index, means.values, "o-", color="k", ms=5)
ax.set_xlabel("Pre-deliberation position rating (1 = strongly disagree, 7 = strongly agree)"); ax.set_ylabel("Position component score")
ax.axhline(0, color="grey", lw=0.8, ls=":")
ax.set_title(f"Cohorts 1-3: r = {fig4a.loc['cohorts_1_3','r']:.2f}  (paper: 0.64)", fontsize=9); plt.tight_layout()

## 3. Fig. 4B check — statement scores relative to the group's opinions (paper: 96% within range)

In [ ]:
fig4b = {c: P.fig4b_within_range(P.select_cohort(opinions, c), P.select_cohort(statements, c)) for c in ["cohorts_1_3", "training", "vca"]}
pd.DataFrame({(c, s): v for c, dd in fig4b.items() for s, v in dd.items()}).T

In [ ]:
d_op = P.select_cohort(opinions, "cohorts_1_3"); d_st = P.select_cohort(statements, "cohorts_1_3")
fig, ax = plt.subplots(figsize=(5, 3.2))
for vals, lab, col in [(d_op["score"], "opinions", "tab:red"), (d_st["initial_score"], "initial statements", "tab:blue"), (d_st["revised_score"], "revised statements", "tab:purple")]:
    ax.hist(vals.dropna(), bins=60, density=True, histtype="step", lw=1.5, label=lab, color=col)
ax.set_xlabel("Position component score (0 = negating, 1 = affirming)"); ax.legend(frameon=False, fontsize=8); plt.tight_layout()

## 4. Fig. 4C — minority weight via convex regression (primary specification)

Main-task cohorts 1–3, pre-registered groups (n = 1047 rounds); minority = smaller side of neutral on the pre-deliberation
rating, neutral opinions kept as non-minority (SM 5.4.1); rounds with a tie or no dissent excluded; columns ordered as in the data.
Analytic SEs are the OLS standard errors of the constrained fit (as in the paper); bootstrap SEs resample rounds.

`assign_minority` is the rule in one place: `side` is the sign of `pre_rating - 4`, `k_min` the
size of the smaller side, `n_div` the group size, and `is_minority` marks the opinions whose weights
are summed. Rounds with no minority (a tie, or unanimity) drop out here.

In [ ]:
op_div = assign_minority(P.select_cohort(opinions, "cohorts_1_3"), neutral="as_majority", ties="exclude", by="rating")
op_div[["launch_id", "round_id", "participant_id", "pre_rating", "side", "is_minority", "k_min", "n_div", "score"]].head()

In [ ]:
op_div.groupby(["n_div", "k_min"])["launch_id"].size().rename("opinions").to_frame().T

In [ ]:
res = P.run_minority_analysis(opinions, candidates, cohort="cohorts_1_3", neutral="as_majority", order="data", n_boot=N_BOOT)
P.save_results(res, f"{OUT_DIR}/fig4c_primary.json")
summary = P.summarize(res); summary.round(3)

In [ ]:
pd.DataFrame(res["contrasts"]).T.round(3)

One convex regression per level of division; the aggregate above averages these, weighted by rounds.

In [ ]:
P.per_level_table(res).round(3)

## 5. Sensitivity analyses
Every combination of the choices the SM leaves open (`P.sensitivity_grid()`): neutral-opinion treatment × basis for the
minority split × column order of the design matrix; model size is swept by running this notebook once per embedding model.
Sample (pre-registered rounds of cohorts 1–3) and tie handling (excluded) are as the SM specifies and are not varied.

In [ ]:
sens = P.run_sensitivity(opinions, candidates, P.sensitivity_grid())
sens.to_csv(f"{OUT_DIR}/sensitivity.csv", index=False); sens.round(3)

## 6. Summary vs paper

The paper's own numbers, read off the position-embedding panel of SM Fig. S60 (the expanded
version of main-text Fig. 4C, per SM 5.2.2), plus the two axis checks it reports. They are literals:
nothing in this repository can recompute them.

In [ ]:
# Paper's values: SM Fig. S60, position-embedding panel (fig4a_r: main-text Fig. 4A; fig4b_within: main-text Fig. 4B).
paper = {"n_rounds": 1047, "minority_share": 0.285, "opinions_sanity": 0.28, "initial_candidates": 0.28, "initial_winner": 0.29,
         "revised_candidates": 0.33, "revised_winner": 0.36, "revised_winner_se": 0.03, "revised_winner_t_vs_true": 2.64,
         "fig4a_r": 0.64, "fig4b_within": 0.96}

In [ ]:
ph = res["phases"]
ours = {"n_rounds": ph["initial_winner"]["n_rounds"], "n_rounds_total": int(P.select_cohort(statements, "cohorts_1_3")["initial_id"].notna().sum()),
        "minority_share": ph["initial_winner"]["true_share"], "opinions_sanity": ph["opinions"]["weight"],
        **{p: ph[p]["weight"] for p in P.PHASES}, "revised_winner_se": ph["revised_winner"]["se"], "revised_winner_t_vs_true": ph["revised_winner"]["t_vs_true"],
        "fig4a_r": fig4a.loc["cohorts_1_3", "r"], "fig4b_within": fig4b["cohorts_1_3"]["both"]["within"], "model": MODEL_TAG, "endpoints": ENDPOINTS, "axis": AXIS_METHOD}
json.dump({"paper": paper, "ours": ours}, open(f"{OUT_DIR}/summary.json", "w"), indent=1, default=float)
pd.DataFrame({"paper": paper, "ours": ours}).round(3)